# Coral Bleaching Map
Data from Harvard Coral Bleaching database [1] shows coral bleaching incidents, with a rating from -1 to 3 indicating unknown / none / medium / high amount of bleaching.

In [1]:
import os
import shutil

# Check if directory is set up properly: If not clone repo
# Check for CoralBleaching.xlsx
if not os.path.exists("./CoralBleaching.xlsx"):
    # Clone the repository if the file is missing
    if not os.path.exists("./coral-bleaching-repo"):
        !git clone https://github.com/danielstebbings/EE581-Project
        %cd ./EE581-Project/Earth_Observation/
        
    
!ls

fatal: destination path 'EE581-Project' already exists and is not an empty directory.
/content/EE581-Project/Earth_Observation
Bleaching_Maps.ipynb  CoralBleaching.xlsx  pyproject.toml  README.md  uv.lock


In [2]:
# Import and authenticate
try:
    import geemap
    import ee
    from dotenv import load_dotenv
    import polars as pl
except:
    !pip install geemap ee dotenv polars fastexcel

# Colab API Key Management
import os

load_dotenv()
EE_KEY = os.getenv('EE_KEY')
EE_PROJECT = os.getenv('EE_PROJECT')

try:
    # Try existing credentials first
    ee.Initialize(project=EE_PROJECT)
    print("✓ Using existing credentials")
except:
    # Authenticate if needed
    print("Authentication required...")
    ee.Authenticate(authorization_code=EE_KEY, quiet=True, code_verifier=None, auth_mode="notebook")
    ee.Initialize(project=EE_PROJECT)
    print("✓ Authentication complete")

✓ Using existing credentials


In [ ]:
# Read Database excel
import polars as pl
bleach_db = pl.read_excel("./CoralBleaching.xlsx")
bleach_db.drop_nulls()
print(len(bleach_db))
print(bleach_db.columns)

print(bleach_db["COUNTRY","SEVERITY_CODE"].sort("SEVERITY_CODE",descending=True))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.9 MB/s eta 0:00:00:00:01
6190
['ID', 'REGION', 'SUBREGION', 'COUNTRY', 'LOCATION', 'LAT', 'LON', 'MONTH', 'YEAR', 'DEPTH', 'SEVERITY_CODE', 'BLEACHING_SEVERITY', 'CORAL_FAMILY', 'CORAL_SPECIES', 'PERCENTAGE_AFFECTED', 'BLEACHING_DURATION', 'MORTALITY_CODE', 'MORTALITY', 'RECOVERY_CODE', 'RECOVERY', 'SURVEY_TYPE', 'SURVEY_AREA', 'WATER_TEMPERATURE', 'OTHER_FACTORS', 'REMARKS', 'SOURCE', 'REFERENCE_CODE', 'COUNTRY_CODE']
shape: (6_190, 2)
┌────────────────────────┬───────────────┐
│ COUNTRY                ┆ SEVERITY_CODE │
│ ---                    ┆ ---           │
│ str                    ┆ i64           │
╞════════════════════════╪═══════════════╡
│ Mexico (Pacific)       ┆ 3             │
│ United Kingdom         ┆ 3             │
│ United Kingdom         ┆ 3             │
│ United Kingdom         ┆ 3             │
│ United Kingdom         ┆ 3             │
│ …                      ┆ …             │
│ Hawaiian Islands (USA) ┆ -

In [ ]:
# Lat Long to EE point
bleach_points_sev = []
bleach_points_med = []
bleach_points_low = []

for i,row in enumerate(bleach_db.iter_rows(named=True)):
    match row["SEVERITY_CODE"]:
        case 3:
            bleach_points_sev.append((row["LON"],row["LAT"]))
        case 2:
            bleach_points_med.append((row["LON"],row["LAT"]))
        case 1:
            bleach_points_low.append((row["LON"],row["LAT"]))

bleach_points_sev = ee.List(bleach_points_sev)
bleach_points_med = ee.List(bleach_points_med)
bleach_points_low = ee.List(bleach_points_low)


def coord2point2feature(point):
    return ee.Feature(ee.Geometry.Point(point))

sev_fc = ee.FeatureCollection(bleach_points_sev.map(coord2point2feature))
med_fc = ee.FeatureCollection(bleach_points_med.map(coord2point2feature))
low_fc = ee.FeatureCollection(bleach_points_low.map(coord2point2feature))

In [ ]:
Bleach_Map = geemap.Map(lite_mode=True,center=(0, 150), zoom=2.5)
Bleach_Map.add_basemap("HYBRID")
Bleach_Map.add_layer(low_fc.draw(color="FFD900", strokeWidth=5), {}, 'Low') 
Bleach_Map.add_layer(med_fc.draw(color="FF8800", strokeWidth=5), {}, 'Medium') 
Bleach_Map.add_layer(sev_fc.draw(color='FF0000', strokeWidth=5), {}, 'Severe') 
Bleach_Map

Map(center=[0, 150], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tran…

# Coral Land Cover Maps

Once we know where an area with bleaching is, we'd like to get ground truth data of where the coral is, as well as the percentage of bleached coral.
Let's look at a specific area of bleaching at 19.082884°S, 148.364868°E in the great barrier reef

We can use the Allen Coral Atlas dataset [2] to show the extent of coral in this area.

In [ ]:
gbr_point = (-19.148578, 148.324081)

gbr_bb = ee.Geometry.BBox(148.287025,-19.177660, 148.351063,-19.124076)

-19.124076, 148.287025
-19.177660, 148.351063

aca = ee.Image("ACA/reef_habitat/v2_0")
aca.getInfo()

coral_map = geemap.Map(lite_mode=True,center=gbr_point,zoom=14)
coral_map.add_basemap("HYBRID")

aca_reef_mask = aca.select('reef_mask').selfMask()
coral_map.addLayer(aca_reef_mask,{},"Reef Extent",False)

aca_geomorphic = aca.select("geomorphic").selfMask()
coral_map.addLayer(aca_geomorphic,{},"Geomorphic Zonation",False)

aca_benthic = aca.select("benthic").selfMask()
coral_map.addLayer(aca_benthic,{}, "Benthic Habitat")

benthic_legend_dict = """
    11	#ffffbe	Sand - Sand is any soft-bottom area dominated by fine unconsolidated sediments.
    12	#e0d05e	Rubble - Rubble is any habitat featuring loose, rough fragments of broken reef material.
    13	#b19c3a	Rock - Rock is any exposed area of hard bare substrate.
    14	#668438	Seagrass - Seagrass is any habitat where seagrass is the dominant biota.
    15	#ff6161	Coral/Algae - Coral/Algae is any hard-bottom area supporting living coral and/or algae.
    18	#9bcc4f	Microalgal Mats - Microalgal Mats are any visible accumulations of microscopic algae in sandy sediments.
    """


coral_map.add_layer(low_fc.draw(color="FFD900", strokeWidth=5), {}, 'Low') 
coral_map.add_layer(med_fc.draw(color="FF8800", strokeWidth=5), {}, 'Medium') 
coral_map.add_layer(sev_fc.draw(color='FF0000', strokeWidth=5), {}, 'Severe') 

coral_map.add_legend(title="Benthic Habitat", legend_dict=geemap.legend_from_ee(benthic_legend_dict))

coral_map

Map(center=[-19.148578, 148.324081], controls=(WidgetControl(options=['position', 'transparent_bg'], position=…

Looking at the same location with Sentinel 2 to recreate this:

In [ ]:

# From sentinel 2 image collection example code
# https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED#colab-python
def mask_s2_clouds(image):
  """Masks clouds in a Sentinel-2 image using the QA band.

  Args:
      image (ee.Image): A Sentinel-2 image.

  Returns:
      ee.Image: A cloud-masked Sentinel-2 image.
  """
  qa = image.select('QA60')

  # Bits 10 and 11 are clouds and cirrus, respectively.
  cloud_bit_mask = 1 << 10
  cirrus_bit_mask = 1 << 11

  # Both flags should be set to zero, indicating clear conditions.
  mask = (
      qa.bitwiseAnd(cloud_bit_mask)
      .eq(0)
      .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
  )

  return image.updateMask(mask).divide(10000)


sent2_ic =  (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(gbr_bb)
    #.filterDate('2024-01-01', '2024-12-31') 
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
    )

sent2_vis = {
    'bands': ['TCI_R', 'TCI_G', 'TCI_B'],
}

s2_map = geemap.Map(center=gbr_point,zoom=14)
s2_map.add_layer(sent2_ic.mean(), sent2_vis, 'Sentinel 2 RGB')
s2_map

Map(center=[-19.148578, 148.324081], controls=(WidgetControl(options=['position', 'transparent_bg'], position=…

In [ ]:
# Download images
sent2_ic.size()

sent_img = sent2_ic.mean().select("TCI_R")
sen2_projection = sent_img.projection().getInfo()
print(sen2_projection)


task = ee.batch.Export.image.toDrive(
    image=sent_img,
    description='imageToDriveExample_transform_tf',
    crs=sen2_projection['crs'],
    crsTransform=sen2_projection['transform'],
    region=gbr_bb,
    fileFormat="TF_RECORD_IMAGE",
)
task.start()

#task = ee.batch.Export.image.toAsset(
    

{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}


# References
[1] ReefBase, “Coral Bleaching Data.” Harvard Dataverse, Feb. 06, 2025. doi: 10.7910/DVN/KUVQKY.

[2] ‘Allen Coral Atlas (ACA) - Geomorphic Zonation and Benthic Habitat - v2.0 | Earth Engine Data Catalog’, Google for Developers. Accessed: Nov. 26, 2025. [Online]. Available: https://developers.google.com/earth-engine/datasets/catalog/ACA_reef_habitat_v2_0
